# Data Processing

## Table of Contents
- [Preliminary Information](#preliminary-information)
- [Purpose](#purpose)

### Preliminary Information

The processing done below consist of [edkidata.jp](https://ekidata.jp/) data only.

The data used is also only their available ***free*** data. Eventually, I will pay for the additional information, but it's unnecessary at the time of writing; see [Purpose](#purpose).

### Purpose

The purpose of this notebook is to determine which stations belong to which lines. This information will be used to optimize PNG resolution where possible.

In [2]:
import numpy as np
import pandas as pd

routes = pd.read_csv("data/Train Station Data - Route.csv")
stations = pd.read_csv("data/Train Station Data - Station.csv")

print(routes.head())
print(stations.head())

   line_cd  company_cd line_name   line_name_k line_name_h line_color_c  \
0     1001           3     中央新幹線   チュウオウシンカンセン       中央新幹線          NaN   
1     1002           3    東海道新幹線  トウカイドウシンカンセン      東海道新幹線          NaN   
2     1003           4     山陽新幹線    サンヨウシンカンセン       山陽新幹線          NaN   
3     1004           2     東北新幹線    トウホクシンカンセン       東北新幹線          NaN   
4     1005           2     上越新幹線   ジョウエツシンカンセン       上越新幹線          NaN   

  line_color_t  line_type         lon        lat  zoom  e_status  e_sort  
0          NaN        NaN  137.493897  35.411438     8         1    1001  
1          NaN        NaN  137.721489  35.144122     7         0    1002  
2          NaN        NaN  133.147896  34.419338     7         0    1003  
3          NaN        NaN  140.763192  38.274267     7         0    1004  
4          NaN        NaN  139.121488  36.798565     8         0    1005  
   station_cd  station_g_cd station_name  station_name_k  station_name_r  \
0     1110101       111

In [3]:
"""
Stations are related to the Route Table through a "line code" (line_cd).

This presumably acts as a foreign key, so a join should work.
"""
merged = stations.merge(
    routes[['line_cd', 'line_name']],
    on='line_cd',
    how='left'
)

joined_data = merged[['line_cd', 'line_name', 'station_cd', 'station_name', 'lat', 'lon']]

In [4]:
"""
Now, we wish to compute each line's "bounding box."

This information will be used to generate smaller PNGs for display, thereby increasing resolution.

Note: To ensure all stations are visible, padding should be added to the bounding box.

---

The above is pretty easy to do, but also consider that we'll need to place the bounding box atop the SVG.

For this, we need to map (lat, lon) -> (svg_x, svg_y).

There's not really a way to get the conversion perfect, but we can try w/ an affine transformation.
"""

def calc_transformation_matrix():
    geo_points = np.array([
        [141.350768, 43.068612], # Sapporo Sta. (lon, lat)
        [139.766103, 35.681391], # Tokyo Sta. (lon, lat)
        [130.460447, 33.542092] # Fukuoka Sta. (lon, lat)
    ])
    
    svg_points = np.array([
        [459.605, 99.773], # Sapporo's pixel position
        [423.699, 303.048], # Tokyo's pixel position
        [236.145, 350.184] # Fukuoka's pixel position
    ])

    # Tranform is 3x2 = [lon_coeff, lat_coeff, offset] for x,y
    A_geo = np.hstack([geo_points, np.ones((len(geo_points), 1))])
    transform, *_ = np.linalg.lstsq(A_geo, svg_points, rcond=None)
    return transform


def geo_to_svg(lon, lat, transform):
    vec = np.array([lon, lat, 1.0])
    x, y = vec @ transform
    return x, y


def svg_to_png_pixel(x, y, bbox, out_w=128, out_h=128):
    min_x, min_y, max_x, max_y = bbox
    svg_width = max_x - min_x
    svg_height = max_y - min_y

    alpha_w = out_w / svg_width
    alpha_h = out_h / svg_height
    alpha = min(alpha_w, alpha_h)

    content_w = svg_width * alpha
    content_h = svg_height * alpha
    offset_x = (out_w - content_w) / 2.0
    offset_y = (out_h - content_h) / 2.0

    px = (x - min_x) * alpha + offset_x
    py = (y - min_y) * alpha + offset_y
    return px, py


def calc_line_bounding_box(df, transform):
    PADDING_FRACTION = 0.05     # % of the bbox's own width/height; const padding could be used, too
    MIN_BOX_SIZE = 10.0         # guards against small boxes

    northernmost_sta = df.loc[df['lat'].idxmax()]
    southernmost_sta = df.loc[df['lat'].idxmin()]
    easternmost_sta = df.loc[df['lon'].idxmax()]
    westernmost_sta = df.loc[df['lon'].idxmin()]

    corners_geo = [
        (westernmost_sta['lon'], northernmost_sta['lat']),
        (easternmost_sta['lon'], northernmost_sta['lat']),
        (easternmost_sta['lon'], southernmost_sta['lat']),
        (westernmost_sta['lon'], southernmost_sta['lat']),
    ]
    corners_svg = [geo_to_svg(lon, lat, transform) for lon, lat in corners_geo]
    xs = [c[0] for c in corners_svg]
    ys = [c[1] for c in corners_svg]

    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    width = max_x - min_x
    height = max_y - min_y

    pad_x = width * PADDING_FRACTION
    pad_y = height * PADDING_FRACTION

    min_x -= pad_x; max_x += pad_x
    min_y -= pad_y; max_y += pad_y

    if (max_x - min_x) < MIN_BOX_SIZE:
        cx = (max_x + min_x) / 2
        min_x, max_x = cx - MIN_BOX_SIZE / 2, cx + MIN_BOX_SIZE / 2
    if (max_y - min_y) < MIN_BOX_SIZE:
        cy = (max_y + min_y) / 2
        min_y, max_y = cy - MIN_BOX_SIZE / 2, cy + MIN_BOX_SIZE / 2

    return f"{min_x:.3f},{min_y:.3f},{max_x:.3f},{max_y:.3f}"


def get_line_summary(df):
    transform = calc_transformation_matrix()
    bbox_str = calc_line_bounding_box(df, transform)
    bbox = tuple(float(v) for v in bbox_str.split(","))

    station_pixels = []
    for _, row in df.iterrows():
        svg_x, svg_y = geo_to_svg(row['lon'], row['lat'], transform)
        png_x, png_y = svg_to_png_pixel(svg_x, svg_y, bbox)
        station_pixels.append({
            "station_cd": row['station_cd'],
            "x": round(png_x),
            "y": round(png_y),
        })

    return pd.Series({
        'stations': list(zip(df['station_cd'], df['station_name'])),
        'bbox': bbox_str,
        'station_pixels': station_pixels
    })


grouped = (
    joined_data
    .groupby(['line_cd', 'line_name'])
    .apply(get_line_summary)
    .reset_index()
)

print(grouped.head())
grouped.to_csv('stations_grouped_by_line.csv', index=False)

   line_cd           line_name  \
0    11101      JR函館本線(函館～長万部)   
1    11102      JR函館本線(長万部～小樽)   
2    11103       JR函館本線(小樽～旭川)   
3    11104  JR室蘭本線(長万部・室蘭～苫小牧)   
4    11105     JR室蘭本線(苫小牧～岩見沢)   

                                            stations  \
0  [(1110101, 函館), (1110102, 五稜郭), (1110103, 桔梗),...   
1  [(1110201, 長万部), (1110202, 二股), (1110203, 蕨岱),...   
2  [(1110301, 小樽), (1110302, 南小樽), (1110303, 小樽築港...   
3  [(1110401, 長万部), (1110402, 静狩), (1110403, 小幌),...   
4  [(1110501, 苫小牧), (1110502, 沼ノ端), (1110503, 遠浅)...   

                              bbox  \
0  436.707,112.741,448.212,136.065   
1   437.655,93.663,453.238,115.766   
2   451.054,78.141,481.565,102.885   
3  438.410,109.875,465.523,121.516   
4   461.737,95.530,471.737,113.128   

                                      station_pixels  
0  [{'station_cd': 1110101, 'x': 85, 'y': 122}, {...  
1  [{'station_cd': 1110201, 'x': 31, 'y': 117}, {...  
2  [{'station_cd': 1110301, 'x': 6, 'y': 86}, {'s...  
3  [{'sta

In [5]:
"""
Generate images for each line's bbox.
"""
from pathlib import Path
import re
import subprocess

__RUST_BINARY = './../../target/debug/jr-hyaku'
__SVG_PATH = './../../assets/Vemaps/jp-02/jp-02.svg'
__OUTPUT_DIR = Path('./data/images')
__OUTPUT_DIR.mkdir(exist_ok=True)

successes = []
failures = []
for _, row in grouped.iterrows():
    line_name = row['line_name']
    bbox = row['bbox']
    out_fp = __OUTPUT_DIR / f"{line_name}.png"

    if line_name in successes:
        print(f"Duplicate detected: {line_name}")
        if line_name not in failures:
            failures.append((line_name, "DUPLICATE"))
        continue

    result = subprocess.run(
        [__RUST_BINARY, __SVG_PATH, "128x128", bbox, str(out_fp)],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(f"FAILED: {line_name}\n  {result.stderr.strip()}")
        failures.append((line_name, result.stderr.strip()))
    else:
        print(f"OK: {line_name} -> {out_fp}")
        successes.append(line_name)

print(f"\nDone. {len(successes)} succeeded, {len(failures)} failed.")

OK: JR函館本線(函館～長万部) -> data/images/JR函館本線(函館～長万部).png
OK: JR函館本線(長万部～小樽) -> data/images/JR函館本線(長万部～小樽).png
OK: JR函館本線(小樽～旭川) -> data/images/JR函館本線(小樽～旭川).png
OK: JR室蘭本線(長万部・室蘭～苫小牧) -> data/images/JR室蘭本線(長万部・室蘭～苫小牧).png
OK: JR室蘭本線(苫小牧～岩見沢) -> data/images/JR室蘭本線(苫小牧～岩見沢).png
OK: JR根室本線(滝川～富良野) -> data/images/JR根室本線(滝川～富良野).png
OK: JR根室本線(新得～釧路) -> data/images/JR根室本線(新得～釧路).png
OK: 花咲線 -> data/images/花咲線.png
OK: JR千歳線 -> data/images/JR千歳線.png
OK: JR石勝線 -> data/images/JR石勝線.png
OK: JR日高本線 -> data/images/JR日高本線.png
OK: JR札沼線 -> data/images/JR札沼線.png
OK: JR留萌本線 -> data/images/JR留萌本線.png
OK: JR富良野線 -> data/images/JR富良野線.png
OK: JR宗谷本線 -> data/images/JR宗谷本線.png
OK: JR石北本線 -> data/images/JR石北本線.png
OK: JR釧網本線 -> data/images/JR釧網本線.png
OK: JR海峡線 -> data/images/JR海峡線.png
OK: JR江差線 -> data/images/JR江差線.png
OK: JR東北本線(八戸～青森) -> data/images/JR東北本線(八戸～青森).png
OK: JR奥羽本線(新庄～青森) -> data/images/JR奥羽本線(新庄～青森).png
OK: はまなすベイライン大湊線 -> data/images/はまなすベイライン大湊線.png
OK: JR五能線 -> data/images/JR五能線.png
OK: JR津軽線